# H&M 2년 M2 체크포인트 진단

기존 H&M 2년 seed 42 체크포인트만 불러와 평가합니다. 모델을 다시 학습하지 않으며 test/holdout도 열지 않습니다.

1. `high/equal/low` 게이트와 실효강도 $\rho$ 곡선을 비교합니다.
2. 선택된 안정 구간에서 `N-only`, `V-only`, `N+V` 축을 분해합니다.
3. M1보다 정확도 가드레일을 유지하면서 가격·구매금액 가중 적중값이 개선되고, shuffled-user와 adapter-only 대조군보다 우월한지 판정합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = '99c191ba9921fd13489db7862bf4b7dc747a07c3'
REPO = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO),
], check=True)
os.chdir(REPO)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert head == REVIEWED_SHA, (head, REVIEWED_SHA)
print('검토된 코드 로드:', head)

In [ ]:
import importlib
import torch
import lightgcn_clv_dual_hm2y_diagnostic as diagnostic

importlib.reload(diagnostic)
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'

SUITE_DIR = Path('/content/drive/MyDrive/논문/data/results_clv_dual_hm2y_suite/hm2y_m2_suite')
candidates = sorted(
    SUITE_DIR.glob('clv_dual_hm2y_suite_*.json'),
    key=lambda path: path.stat().st_mtime,
)
assert candidates, f'H&M 2년 suite 결과 JSON이 없습니다: {SUITE_DIR}'
SUITE_RESULT_JSON = candidates[-1]

cfg = diagnostic.configure_hm2y_diagnostic(
    SUITE_RESULT_JSON,
    out_dir=SUITE_DIR / 'checkpoint_diagnostics',
)
print('입력 결과:', SUITE_RESULT_JSON)
print('게이트:', cfg.gate_shapes)
print('실효강도 rho:', cfg.rho_grid)
print('축:', cfg.axis_modes)
print('학습 실행:', False, '| test:', cfg.eval_test, '| holdout:', cfg.eval_holdout)

## 체크포인트 진단 실행

아래 셀은 재학습 없이 전체 카탈로그 순위를 평가합니다. 각 평가점이 끝날 때마다 진행상황이 출력됩니다.

In [ ]:
result_df = diagnostic.run_hm2y_diagnostic(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

paths = result_df.attrs['result_paths']
decision = result_df.attrs['decision']
print('최종 판정:', decision['success'])
print('사유:', decision['reason'])
print('선택 게이트/rho:', decision.get('selected_gate_shape'), decision.get('selected_rho'))

print('\n게이트×rho 판정표')
display(pd.read_csv(paths['decision_csv']).sort_values(['gate_shape', 'rho']))
print('\nN/V 축 분해표')
display(pd.read_csv(paths['axis_csv']).sort_values(['model_id', 'axis_mode']))
print('\n전체 곡선')
display(result_df.sort_values(['model_id', 'gate_shape', 'rho']))
print('\n결과 파일')
for label, path in paths.items():
    print(f' - {label}: {path}')